# HinglishSense: fine-tune XLM-RoBERTa on Google Colab (free T4 GPU)

This notebook fine-tunes **`xlm-roberta-base`**, a multilingual transformer, on our Hinglish sentiment data.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

**Steps:** 1) install libraries, 2) upload `train.csv`, `val.csv` and `test.csv` from `data/processed/`, 3) train, 4) evaluate, 5) download the model.

Settings (same as `src/config.py`): 3 epochs, learning rate 2e-5, batch size 16, max length 64, fp16. The best epoch is chosen by **validation macro-F1**.

## 1. Install libraries and check the GPU

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Upload `train.csv`, `val.csv`, `test.csv` (from `data/processed/`)

In [ ]:
import os
from google.colab import files

needed = ["train.csv", "val.csv", "test.csv"]
if not all(os.path.exists(name) for name in needed):
    uploaded = files.upload()  # choose the three CSV files
print({name: os.path.exists(name) for name in needed})

In [ ]:
import pandas as pd

LABELS = ["negative", "neutral", "positive"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}

frames = {}
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"{split}.csv", dtype=str, keep_default_na=False)
    df["labels"] = df["label"].map(LABEL_TO_ID)
    frames[split] = df
    print(split, len(df), df["label"].value_counts().to_dict())

## 3. Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 64
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN, padding="max_length")

datasets = {}
for split, df in frames.items():
    datasets[split] = Dataset.from_pandas(df[["text", "labels"]]).map(tokenize, batched=True)

## 4. Train (about 15-25 minutes on a T4)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label=dict(enumerate(LABELS)), label2id=LABEL_TO_ID)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions),
            "macro_f1": f1_score(labels, predictions, average="macro")}

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(), seed=42, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=datasets["train"],
                  eval_dataset=datasets["val"], compute_metrics=compute_metrics)
trainer.train()

## 5. Evaluate on the test set

In [ ]:
from sklearn.metrics import classification_report

output = trainer.predict(datasets["test"])
test_predictions = np.argmax(output.predictions, axis=-1)
print(classification_report(output.label_ids, test_predictions, target_names=LABELS, digits=4))
print("Test macro-F1:", round(f1_score(output.label_ids, test_predictions, average="macro"), 4))
print("Test accuracy:", round(accuracy_score(output.label_ids, test_predictions), 4))

## 6. Save and download the model

Unzip it into `models/transformer/` in the project folder. The Streamlit app will then offer the *transformer* option.

In [ ]:
import json, shutil

trainer.save_model("transformer")
tokenizer.save_pretrained("transformer")
with open("transformer/metrics.json", "w") as f:
    json.dump({"test_macro_f1": f1_score(output.label_ids, test_predictions, average="macro"),
               "test_accuracy": accuracy_score(output.label_ids, test_predictions)}, f, indent=2)
shutil.make_archive("transformer", "zip", "transformer")
files.download("transformer.zip")